# 4장. 학습이 불안정해지는 이유

손실이 줄지 않을 때, 신경망의 어디에서 무엇을 재야 원인을 알 수 있는가.

**준비:** 한국전력거래소 「시간별 전국 전력수요량」 CSV 두 개(2023년, 2025년)와 기상청 종관기상관측(ASOS) 시간자료 CSV 두 개(2023년, 2025년)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 끝까지 실행하는 데 10분쯤 걸린다.

## 4.1 입력을 표준화하지 않고 학습시키기

In [ ]:
import glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def first_line(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        return f.readline()

csvs = sorted(glob.glob("*.csv"))
power_files = [f for f in csvs if first_line(f).startswith("날짜,1시")]
weather_files = [f for f in csvs if first_line(f).startswith("지점,지점명")]

wide = pd.concat([pd.read_csv(f, encoding="cp949") for f in power_files]).dropna(subset=["날짜"])
long = wide.melt("날짜", var_name="시", value_name="MWh")
long["시각"] = pd.to_datetime(long["날짜"]) + pd.to_timedelta(long["시"].str.replace("시", "").astype(int), unit="h")
power = long.set_index("시각")["MWh"].sort_index() / 1000          # GW

FIVE = ["서울", "대전", "대구", "부산", "광주"]
w = pd.concat([pd.read_csv(f, encoding="cp949", usecols=["지점명", "일시", "기온(°C)"]) for f in weather_files])
w["일시"] = pd.to_datetime(w["일시"])
temp = w[w["지점명"].isin(FIVE)].pivot(index="일시", columns="지점명", values="기온(°C)").interpolate()

h = pd.DataFrame({"GW": power, "기온": temp.mean(axis=1)}).dropna()
h["날짜"] = (h.index - pd.Timedelta(hours=1)).normalize()
d = h.groupby("날짜").agg(GW=("GW", "mean"), 기온=("기온", "mean"))

HOLIDAYS = pd.to_datetime([
    "2023-01-01", "2023-01-21", "2023-01-22", "2023-01-23", "2023-01-24", "2023-03-01",
    "2023-05-05", "2023-05-27", "2023-05-29", "2023-06-06", "2023-08-15", "2023-09-28",
    "2023-09-29", "2023-09-30", "2023-10-02", "2023-10-03", "2023-10-09", "2023-12-25",
    "2025-01-01", "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30", "2025-03-01",
    "2025-03-03", "2025-05-05", "2025-05-06", "2025-06-03", "2025-06-06", "2025-08-15",
    "2025-10-03", "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09",
    "2025-12-25"])
weekday = d[(d.index.dayofweek < 5) & ~d.index.isin(HOLIDAYS)]
T23, Y23 = weekday.loc["2023", "기온"].to_numpy(), weekday.loc["2023", "GW"].to_numpy()
T25, Y25 = weekday.loc["2025", "기온"].to_numpy(), weekday.loc["2025", "GW"].to_numpy()
print(len(Y23), len(Y25))

In [ ]:
import torch
from torch import nn

def rmse(actual, predicted):
    return float(np.sqrt(((actual - predicted) ** 2).mean()))

def column(values):
    """넘파이 배열을 (자료의 수, 1) 모양의 32비트 텐서로 바꾼다."""
    return torch.tensor(np.asarray(values)[:, None], dtype=torch.float32)

def pad(text, width):
    """글자 수를 맞춰 오른쪽을 공백으로 채운다. 한글은 두 칸으로 센다."""
    wide = sum(ord(ch) > 0x1100 for ch in text)
    return text + " " * max(width - len(text) - wide, 1)

def nums(values, digits=4):
    """텐서나 배열의 값을 반올림한 파이썬 숫자의 목록으로 바꾼다."""
    if torch.is_tensor(values):
        values = values.detach()
    return [round(float(v), digits) for v in values]

torch.set_num_threads(1)                 # 작은 텐서에서는 스레드 하나가 더 빠르다
t_mean, t_std = T23.mean(), T23.std()
y_mean, y_std = Y23.mean(), Y23.std()
loss_fn = nn.MSELoss()
print(round(t_mean, 2), round(t_std, 2), round(y_mean, 2), round(y_std, 2))
print(round(T23.min(), 1), round(T23.max(), 1))
print(round(Y23.min(), 1), round(Y23.max(), 1))

In [ ]:
def fit_small(scale_x, scale_y, rate, seed, steps=3000):
    """유닛 8개인 신경망. scale_x, scale_y는 입력과 정답을 표준화할지 정한다."""
    x_shift, x_unit = (t_mean, t_std) if scale_x else (0.0, 1.0)
    y_shift, y_unit = (y_mean, y_std) if scale_y else (0.0, 1.0)
    X, Y = column((T23 - x_shift) / x_unit), column((Y23 - y_shift) / y_unit)
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(1, 8), nn.ReLU(), nn.Linear(8, 1))
    optimizer = torch.optim.SGD(model.parameters(), lr=rate)
    trace = []
    for step in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(model(X), Y)
        loss.backward()
        optimizer.step()
        trace.append(float(np.sqrt(loss.item())) * y_unit)     # 2023년 RMSE(GW)
        if not np.isfinite(trace[-1]):
            return trace, float("nan")                          # 발산
    with torch.no_grad():
        out = model(column((T25 - x_shift) / x_unit)).numpy()[:, 0]
    return trace, rmse(Y25, out.astype(np.float64) * y_unit + y_shift)

cases = {"둘 다 그대로": (False, False), "입력만 표준화": (True, False),
         "정답만 표준화": (False, True), "둘 다 표준화": (True, True)}
rates = [0.05, 0.01, 0.001, 0.0001]
print(pad("학습률", 14) + "".join(f"{rate:>14}" for rate in rates))
small = {}
for name, (scale_x, scale_y) in cases.items():
    row = pad(name, 14)
    for rate in rates:
        marks = [fit_small(scale_x, scale_y, rate, seed)[1] for seed in range(5)]
        small[name, rate] = marks
        alive = [mark for mark in marks if np.isfinite(mark)]
        middle = f"{np.median(alive):.2f}" if alive else "-"
        row += f"{middle:>9} ({5 - len(alive)}) "
    print(row)

In [ ]:
best = [("둘 다 그대로", 0.0001), ("입력만 표준화", 0.001), ("둘 다 표준화", 0.05)]
for name, rate in best:
    print(name, rate, [round(mark, 2) for mark in small[name, rate]])

In [ ]:
shown = [("둘 다 그대로", 0.001), ("둘 다 그대로", 0.0001), ("둘 다 그대로", 0.00001),
         ("입력만 표준화", 0.001), ("둘 다 표준화", 0.05)]
curves = {}
for name, rate in shown:
    trace, mark = fit_small(*cases[name], rate, seed=3)
    curves[name, rate] = trace
    end = "발산" if not np.isfinite(mark) else f"{trace[-1]:.2f} {mark:.2f}"
    print(pad(name, 14), f"{rate:<8g}", len(trace), end)

In [ ]:
def curvature(x):
    """직선 모델 w·x + c의 평균 제곱 오차를 w, c로 두 번 미분한 행렬과 그 고윳값."""
    design = np.column_stack([x, np.ones(len(x))])
    hessian = 2 * design.T @ design / len(x)
    return hessian, np.linalg.eigvalsh(hessian)

for name, x in [("그대로", T23), ("표준화", (T23 - t_mean) / t_std)]:
    hessian, eigen = curvature(x)
    print(name, (hessian.round(2) + 0.0).tolist(), eigen.round(3).tolist())
    print("   ", round(eigen[1] / eigen[0], 1), round(2 / eigen[1], 5))

In [ ]:
def line_descent(x, y, rate, steps=200):
    """직선 모델을 경사하강법으로 학습시킨 뒤의 손실."""
    w, c = 0.0, 0.0
    for _ in range(steps):
        error = w * x + c - y
        w, c = w - rate * 2 * (error * x).mean(), c - rate * 2 * error.mean()
    return ((w * x + c - y) ** 2).mean()

for rate in [0.0030, 0.0031, 0.0032, 0.0033]:
    print(rate, f"{line_descent(T23, Y23, rate):.4g}")

## 4.2 층마다 출력과 기울기의 크기 재기

In [ ]:
X, Y = column((T23 - t_mean) / t_std), column((Y23 - y_mean) / y_std)
X25 = column((T25 - t_mean) / t_std)

def build(depth, act=nn.ReLU, std=None, seed=0, width=32):
    """은닉층이 depth개인 신경망. std가 None이면 파이토치의 기본 시작값을 쓴다.
    std가 함수이면 가중치를 평균 0, 표준편차 std(입력의 수)인 정규분포에서 뽑는다."""
    torch.manual_seed(seed)
    layers, m = [], 1
    for size in [width] * depth + [1]:
        linear = nn.Linear(m, size)
        if std is not None:
            nn.init.normal_(linear.weight, 0.0, std(m))
            nn.init.zeros_(linear.bias)
        layers += [linear, act()]
        m = size
    return nn.Sequential(*layers[:-1])            # 마지막 층 뒤에는 활성화 함수가 없다

def measure(model):
    """은닉층마다 '날 사이의 출력 차이'와 '가중치 기울기의 크기'를 잰다."""
    spread, h = [], X
    for layer in model:
        h = layer(h)
        if not isinstance(layer, (nn.Linear, nn.BatchNorm1d)):
            spread.append(h.std(dim=0).mean().item())      # 유닛마다 247일의 표준편차
    model.zero_grad()
    loss_fn(h, Y).backward()
    grads = [layer.weight.grad.abs().mean().item()
             for layer in model if isinstance(layer, nn.Linear)]
    return spread, grads

PICKS = (1, 5, 10, 15, 20)
HEAD = pad("층", 20) + "".join(f"{k:>10}" for k in PICKS)

def show(name, values):
    print(pad(name, 20) + "".join(f"{values[k - 1]:>10.1e}" for k in PICKS))

model = build(20)
print(len(model), sum(param.numel() for param in model.parameters()))

In [ ]:
print(HEAD)
for name, act in [("ReLU", nn.ReLU), ("시그모이드", nn.Sigmoid), ("tanh", nn.Tanh)]:
    spread, grads = measure(build(20, act))
    show(name + " 출력", spread)
    show(name + " 기울기", grads)

In [ ]:
z = torch.tensor([-6.0, -2.0, 0.0, 2.0, 6.0], requires_grad=True)
s = torch.sigmoid(z)
s.sum().backward()
print(nums(s))
print(nums(z.grad))
print(round(0.25 ** 20, 15), f"{0.25 ** 20:.1e}")

In [ ]:
def train(model, rate=0.02, steps=2000):
    """전체 배치 경사하강법. 2023년의 손실과 2025년 RMSE를 돌려준다. 발산하면 nan."""
    optimizer = torch.optim.SGD(model.parameters(), lr=rate)
    model.train()
    for step in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(model(X), Y)
        loss.backward()
        optimizer.step()
        if not torch.isfinite(loss):
            return float("nan"), float("nan")
    model.eval()
    with torch.no_grad():
        out = model(X25).numpy()[:, 0].astype(np.float64)
        return loss_fn(model(X), Y).item(), rmse(Y25, out * y_std + y_mean)

def report(name, runs):
    """시드마다의 (손실, 2025년 RMSE)를 한 줄로 출력한다."""
    losses = " ".join(f"{loss:.4f}" for loss, mark in runs)
    print(pad(name, 26) + losses + " | " + " ".join(f"{mark:.2f}" for loss, mark in runs))

for rate in [0.2, 0.02, 0.002]:
    runs = [train(build(20, seed=seed), rate) for seed in range(5)]
    report(f"ReLU 20층, 학습률 {rate}", runs)
runs = [train(build(5, nn.Sigmoid, seed=seed)) for seed in range(5)]
report("시그모이드 5층", runs)
runs = [train(build(1, seed=seed)) for seed in range(5)]
report("ReLU 1층", runs)

## 4.3 시작값의 표준편차 정하기

In [ ]:
torch.manual_seed(0)
linear = nn.Linear(300, 200)
weight, bias = linear.weight.detach(), linear.bias.detach()
print(round(weight.std().item(), 4), round(1 / np.sqrt(3 * 300), 4))
print(round(weight.abs().max().item(), 4), round(bias.abs().max().item(), 4),
      round(1 / np.sqrt(300), 4))

In [ ]:
torch.manual_seed(0)
m, k = 1000, 1000
a = torch.relu(torch.randn(5000, m))          # 앞 층의 출력. ReLU를 지난 값이다
print("앞 층 출력의 제곱 평균", round((a ** 2).mean().item(), 3))
choices = [("0.01", 0.01), ("1/sqrt(3m)", 1 / np.sqrt(3 * m)), ("1/sqrt(m)", 1 / np.sqrt(m)),
           ("sqrt(2/m)", np.sqrt(2 / m)), ("1", 1.0)]
for name, s in choices:
    z = a @ (torch.randn(m, k) * s)
    after = (torch.relu(z) ** 2).mean().item()
    formula = m * s ** 2 * 0.5
    print(f"{name:<11} {z.var().item():>12.4g} {formula:>12.4g} {after:>12.4g}")

In [ ]:
inits = {"0.01": lambda m: 0.01, "기본값": None, "1/sqrt(m)": lambda m: 1 / np.sqrt(m),
         "sqrt(2/m)": lambda m: np.sqrt(2 / m), "1": lambda m: 1.0}
print(HEAD)
for name, std in inits.items():
    spread, grads = measure(build(20, std=std))
    show(name + " 출력", spread)
    show(name + " 기울기", grads)

In [ ]:
for name, std in inits.items():
    spread, grads = measure(build(20, std=std))
    measured = (spread[19] / spread[1]) ** (1 / 18)     # 2층에서 20층까지, 한 층에 몇 배
    variance = 32 * std(32) ** 2 if std else 32 / (3 * 32)   # m × Var(w)
    print(pad(name, 12), f"{measured:>8.3f} {np.sqrt(variance / 2):>8.3f}")

In [ ]:
grid = {}
print(pad("은닉층의 수", 12) + "".join(" " * 3 + pad(name, 11) for name in inits))
for depth in [1, 5, 10, 20]:
    row = pad(str(depth), 12)
    for name, std in inits.items():
        runs = [train(build(depth, std=std, seed=seed)) for seed in range(5)]
        grid[depth, name] = runs
        marks = sorted(mark for loss, mark in runs if np.isfinite(mark))
        failed = 5 - len(marks)
        text = f"{marks[len(marks) // 2]:.2f}" if marks else "-"
        row += f"{text:>7} ({failed})   "
    print(row)

In [ ]:
for name in inits:
    report(f"20층, {name}", grid[20, name])

## 4.4 배치 정규화로 층의 입력 크기 맞추기

In [ ]:
z = torch.tensor([[2.0], [4.0], [9.0]])
norm = nn.BatchNorm1d(1)
print(norm.weight.item(), norm.bias.item())
print(norm.running_mean.item(), norm.running_var.item())
norm.train()
print(nums(norm(z)[:, 0]))
mean, var = z.mean().item(), z.var(correction=0).item()
print(round(mean, 4), round(var, 4), nums((z[:, 0] - mean) / np.sqrt(var + 1e-5)))
print(round(norm.running_mean.item(), 4), round(norm.running_var.item(), 4),
      round(z.var(correction=1).item(), 4))
norm.eval()
print(nums(norm(z)[:, 0]))

In [ ]:
def build_bn(depth, seed=0, width=32):
    """build와 같고, 은닉층마다 Linear와 ReLU 사이에 배치 정규화 층을 넣는다."""
    torch.manual_seed(seed)
    layers, m = [], 1
    for _ in range(depth):
        layers += [nn.Linear(m, width), nn.BatchNorm1d(width), nn.ReLU()]
        m = width
    return nn.Sequential(*layers, nn.Linear(m, 1))

print(HEAD)
for name, model in [("기본값", build(20)), ("기본값+정규화", build_bn(20))]:
    spread, grads = measure(model)
    show(name + " 출력", spread)
    show(name + " 기울기", grads)

In [ ]:
bn_models = [build_bn(20, seed=seed) for seed in range(5)]
bn_runs = [train(model) for model in bn_models]
report("20층, 기본값+정규화", bn_runs)

In [ ]:
def predict(model, x):
    with torch.no_grad():
        return model(x).numpy()[:, 0].astype(np.float64) * y_std + y_mean

model = bn_models[0]
hot = np.argsort(T25)[-32:]                       # 2025년에서 가장 더운 32일
model.eval()
whole_eval, hot_eval = predict(model, X25), predict(model, X25[hot])
model.train()
whole_train, hot_train = predict(model, X25), predict(model, X25[hot])
model.eval()
print("평가 모드, 243일:", f"{rmse(Y25, whole_eval):.2f}")
print("학습 모드, 243일:", f"{rmse(Y25, whole_train):.2f}")
print("평가 모드, 더운 32일만:", f"{rmse(Y25[hot], hot_eval):.2f}",
      f"{np.abs(hot_eval - whole_eval[hot]).max():.4f}")
print("학습 모드, 더운 32일만:", f"{rmse(Y25[hot], hot_train):.2f}",
      f"{np.abs(hot_train - whole_eval[hot]).max():.2f}")
print(round(T25[hot].min(), 1), round(T25[hot].max(), 1),
      round(hot_eval.min(), 1), round(hot_eval.max(), 1),
      round(hot_train.min(), 1), round(hot_train.max(), 1))
grid_t = np.linspace(-10, 31, 411)                # 그림 4-7에서 쓴다
curve_bn = predict(model, column((grid_t - t_mean) / t_std))

In [ ]:
model.train()
try:
    model(X25[:1])
except ValueError as error:
    print(str(error)[:62])
model.eval()
print(round(predict(model, X25[:1])[0], 2), round(Y25[0], 2), round(T25[0], 1))

In [ ]:
first = model[1]                                  # 첫 배치 정규화 층
before = first.running_mean[:3].clone()
model.train()
for _ in range(20):
    predict(model, X25[hot])                      # 학습 모드에서 예측만 20번 한다
model.eval()
print(nums(before, 3), nums(first.running_mean[:3], 3))
print(f"{rmse(Y25, predict(model, X25)):.2f}")

## 4.5 잔차 연결로 깊은 신경망 학습시키기

In [ ]:
class Block(nn.Module):
    """잔차 블록. 들어온 값 h에, 층 두 개를 지난 값을 더해서 내보낸다."""

    def __init__(self, width, zero):
        super().__init__()
        self.inner = nn.Linear(width, width)
        self.act = nn.ReLU()
        self.outer = nn.Linear(width, width)
        if zero:                                  # 가지의 마지막 층을 0으로 시작한다
            nn.init.zeros_(self.outer.weight)
            nn.init.zeros_(self.outer.bias)

    def forward(self, h):
        return h + self.outer(self.act(self.inner(h)))

def build_res(blocks, zero=True, seed=0, width=32):
    """잔차 블록이 blocks개인 신경망. Linear 층은 2 × blocks + 2개다."""
    torch.manual_seed(seed)
    body = [Block(width, zero) for _ in range(blocks)]
    return nn.Sequential(nn.Linear(1, width), *body, nn.Linear(width, 1))

model = build_res(10)
print(len(model), sum(isinstance(layer, nn.Linear) for layer in model.modules()),
      sum(param.numel() for param in model.parameters()))

In [ ]:
def size_along(model):
    """블록을 하나 지날 때마다 값의 크기(제곱 평균의 제곱근)를 잰다."""
    sizes, h = [], X
    for layer in model:
        h = layer(h)
        sizes.append(h.pow(2).mean().sqrt().item())
    return sizes

def first_grad(model):
    """첫 층 가중치의 기울기의 크기."""
    model.zero_grad()
    loss_fn(model(X), Y).backward()
    return model[0].weight.grad.abs().mean().item()

pair = [("0으로 시작", build_res(25)), ("기본값으로 시작", build_res(25, zero=False))]
for name, model in pair:
    sizes = size_along(model)
    print(name, [f"{sizes[k]:.2g}" for k in (0, 5, 10, 15, 20, 25)], f"{sizes[26]:.2g}")
print("첫 층의 기울기")
for depth, blocks in [(10, 4), (20, 9), (50, 24)]:
    models = [build(depth), build(depth, std=lambda m: np.sqrt(2 / m)), build_bn(depth),
              build_res(blocks), build_res(blocks, zero=False)]
    print(depth, " ".join(f"{first_grad(model):.1e}" for model in models))

In [ ]:
he = lambda m: np.sqrt(2 / m)
final = {(20, name): grid[20, name] for name in ("기본값", "1/sqrt(m)", "sqrt(2/m)")}
final[20, "기본값+정규화"] = bn_runs
half = inits["1/sqrt(m)"]
makers = {"1/sqrt(m)": lambda depth, seed: build(depth, std=half, seed=seed),
          "sqrt(2/m)": lambda depth, seed: build(depth, std=he, seed=seed),
          "기본값+정규화": lambda depth, seed: build_bn(depth, seed=seed),
          "잔차(기본값 시작)": lambda d, seed: build_res(d // 2 - 1, False, seed),
          "잔차(0 시작)": lambda d, seed: build_res(d // 2 - 1, True, seed)}
for depth in [20, 50]:
    for name, make in makers.items():
        if (depth, name) not in final:
            final[depth, name] = [train(make(depth, seed)) for seed in range(5)]
for (depth, name), runs in sorted(final.items(), key=lambda item: item[0][0]):
    report(f"{depth}층, {name}", runs)

In [ ]:
losses = []
for count in [1, 2]:
    torch.set_num_threads(count)
    losses.append(train(build_bn(20), steps=500)[0])
torch.set_num_threads(1)
print(f"{losses[0]:.6f}", f"{losses[1]:.6f}")

In [ ]:
import sys, platform

print(sys.version.split()[0], np.__version__, torch.__version__)
print(platform.machine(), torch.get_default_dtype(), torch.get_num_threads())

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 4-1
fig, ax = plt.subplots(figsize=(7, 4.8))
steps = np.arange(1, 3001)
looks = [(BRICK, (0, (1.5, 1.5)), 1.4), (BRICK, "-", 1.8), (BRICK, (0, (5, 2)), 1.4), (NAVY, "-", 1.6), (TEAL, "-", 2.4)]
for (name, rate), (col, ls, lw) in zip(shown, looks):
    trace = curves[name, rate]
    tail = "발산" if len(trace) < 3000 else f"3,000걸음 뒤 {trace[-1]:.2f}GW"
    digits = f"{rate:.5f}".rstrip("0")
    ax.plot(steps[:len(trace)], np.minimum(trace, 1e4), color=col, ls=ls, lw=lw, label=f"{name}, 학습률 {digits}: {tail}")
ax.axhline(y_std, color=GRAY, lw=0.9, ls=(0, (2, 2)))
ax.text(1.1, y_std * 0.9, f"회색 점선: 평균으로 예측할 때 {y_std:.2f}GW", fontsize=9, color="#5b6470", ha="left", va="top")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(1, 3000); ax.set_ylim(2, 8000)
plain_ticks(ax.xaxis); ax.set_yticks([2, 5, 10, 20, 50, 100, 200, 500, 1000])
ax.yaxis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:g}")); ax.minorticks_off()
ax.set_xlabel("가중치를 고친 횟수 (로그 축)", fontsize=9)
ax.set_ylabel("2023년 자료의 RMSE (GW, 로그 축)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_4_1.png", dpi=300)
plt.show()

In [ ]:
# 그림 4-3
sci = FuncFormatter(lambda val, _: f"{val:.0e}")
fig, axes = plt.subplots(2, 1, figsize=(7, 8.0))
for name, act, col, ls in [("ReLU", nn.ReLU, NAVY, "-"), ("tanh", nn.Tanh, TEAL, (0, (5, 2))), ("시그모이드", nn.Sigmoid, BRICK, "-")]:
    spread, grads = measure(build(20, act))
    axes[0].plot(range(1, 21), np.maximum(spread, 1e-12), color=col, ls=ls, lw=1.8, marker="o", ms=3.4, label=name)
    axes[1].plot(range(1, 22), grads, color=col, ls=ls, lw=1.8, marker="o", ms=3.4, label=name)
    axes[0].text(20.4, max(spread[-1], 1e-12), f"{spread[-1]:.1e}", fontsize=9, color=col, va="center" if spread[-1] > 1e-12 else "bottom", weight="bold")
    axes[1].text(0.6, grads[0], f"{grads[0]:.1e}", fontsize=9, color=col, ha="right", va="center", weight="bold")
axes[0].set_yscale("log"); axes[0].set_ylim(1e-12, 3); axes[0].set_yticks([1e-12, 1e-10, 1e-8, 1e-6, 1e-4, 1e-2, 1])
axes[0].set_xlabel("은닉층의 번호 (1이 입력 쪽, 20이 출력 쪽)", fontsize=9)
axes[0].set_ylabel("날 사이의 출력 차이 (로그 축)", fontsize=9)
axes[1].set_yscale("log"); axes[1].set_ylim(1e-20, 1); axes[1].set_yticks([1e-20, 1e-16, 1e-12, 1e-8, 1e-4, 1])
axes[1].set_xlabel("층의 번호 (1이 입력 쪽, 21이 출력층)", fontsize=9)
axes[1].set_ylabel("가중치 기울기의 크기 (로그 축)", fontsize=9)
for ax in axes:
    ax.set_xticks([1, 5, 10, 15, 20]); ax.yaxis.set_major_formatter(sci); ax.minorticks_off()
    ax.legend(frameon=False, fontsize=9, loc="lower left" if ax is axes[0] else "lower right")
    frame(ax, "① 층마다 남은 출력 차이 (오른쪽 숫자는 20번째 층)" if ax is axes[0] else "② 층마다 받은 기울기 (왼쪽 숫자는 첫 층)")
axes[0].set_xlim(0, 23.5); axes[1].set_xlim(-2.6, 22)
fig.tight_layout(h_pad=2.2)
fig.savefig("fig_4_3.png", dpi=300)
plt.show()

In [ ]:
# 그림 4-5
looks = {"0.01": (GRAY, (0, (5, 2)), 1.4), "기본값": (BRICK, "-", 1.8), "1/sqrt(m)": (NAVY, (0, (5, 2)), 1.6),
         "sqrt(2/m)": (TEAL, "-", 2.4), "1": (NAVY, (0, (1.5, 1.5)), 1.4)}
fig, axes = plt.subplots(2, 1, figsize=(7, 8.0))
for name, std in inits.items():
    col, ls, lw = looks[name]
    spread, grads = measure(build(20, std=std))
    axes[0].plot(range(1, 21), np.clip(spread, 1e-16, 1e12), color=col, ls=ls, lw=lw, marker="o", ms=3, label=name)
    axes[1].plot(range(1, 22), np.clip(grads, 1e-32, 1e24), color=col, ls=ls, lw=lw, marker="o", ms=3, label=name)
    axes[0].text(20.4, np.clip(spread[-1], 1e-16, 1e12), f"{spread[-1]:.1e}", fontsize=9, color=col, va="center" if spread[-1] > 1e-16 else "bottom", weight="bold")
    axes[1].text(0.6, np.clip(grads[0], 1e-32, 1e24), f"{grads[0]:.1e}", fontsize=9, color=col, ha="right", va="center", weight="bold")
axes[0].set_yscale("log"); axes[0].set_ylim(1e-16, 1e12); axes[0].set_yticks([1e-16, 1e-12, 1e-8, 1e-4, 1, 1e4, 1e8, 1e12])
axes[0].set_xlabel("은닉층의 번호 (1이 입력 쪽, 20이 출력 쪽)", fontsize=9)
axes[0].set_ylabel("날 사이의 출력 차이 (로그 축)", fontsize=9)
axes[1].set_yscale("log"); axes[1].set_ylim(1e-32, 1e24); axes[1].set_yticks([1e-32, 1e-24, 1e-16, 1e-8, 1, 1e8, 1e16, 1e24])
axes[1].set_xlabel("층의 번호 (1이 입력 쪽, 21이 출력층)", fontsize=9)
axes[1].set_ylabel("가중치 기울기의 크기 (로그 축)", fontsize=9)
for ax in axes:
    ax.set_xticks([1, 5, 10, 15, 20]); ax.yaxis.set_major_formatter(sci); ax.minorticks_off()
    frame(ax, "① 층마다 남은 출력 차이 (오른쪽 숫자는 20번째 층)" if ax is axes[0] else "② 층마다 받은 기울기 (왼쪽 숫자는 첫 층)")
axes[0].set_xlim(0, 23.5); axes[1].set_xlim(-2.6, 22)
axes[1].legend(frameon=False, fontsize=9, loc="center", bbox_to_anchor=(0.55, 0.66), ncol=5, title="시작값의 표준편차", title_fontsize=9)
fig.tight_layout(h_pad=2.2)
fig.savefig("fig_4_5.png", dpi=300)
plt.show()

In [ ]:
# 그림 4-7
fig, ax = plt.subplots(figsize=(7, 4.8))
ax.scatter(T25, Y25, s=10, color=to_rgba(GRAY, 0.6), edgecolor="none", label="2025년 평일 243일의 실제 수요")
ax.plot(grid_t, curve_bn, color=TEAL, lw=2.0, label="평가 모드의 예측")
ax.scatter(T25[hot], hot_train, s=22, color=BRICK, edgecolor="white", lw=0.5, zorder=5, label="학습 모드, 가장 더운 32일만 넣은 예측")
ax.set_xlim(-10.5, 31.5); ax.set_ylim(50, 94); ax.set_xticks(range(-10, 31, 5))
ax.set_xlabel("일평균 기온(°C)", fontsize=9)
ax.set_ylabel("일평균 전력수요(GW)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper center")
frame(ax)
fig.tight_layout()
fig.savefig("fig_4_7.png", dpi=300)
plt.show()

In [ ]:
# 그림 4-9
depths = [4, 6, 10, 14, 20, 30, 40, 50]
by_depth = {"기본값": [first_grad(build(d)) for d in depths],
          "sqrt(2/m)": [first_grad(build(d, std=he)) for d in depths],
          "기본값+정규화": [first_grad(build_bn(d)) for d in depths],
          "잔차(0 시작)": [first_grad(build_res(d // 2 - 1)) for d in depths]}
looks = {"기본값": (BRICK, "-", 1.8), "sqrt(2/m)": (NAVY, (0, (5, 2)), 1.6), "기본값+정규화": (GRAY, "-", 1.6),
         "잔차(0 시작)": (TEAL, "-", 2.4)}
fig, ax = plt.subplots(figsize=(7, 4.8))
for name, values in by_depth.items():
    col, ls, lw = looks[name]
    ax.plot(depths, values, color=col, ls=ls, lw=lw, marker="o", ms=4, label=name)
    ax.text(51.2, values[-1], f"{values[-1]:.1e}", fontsize=9, color=col, weight="bold", va={NAVY: "bottom", TEAL: "top"}.get(col, "center"))
ax.set_yscale("log"); ax.set_ylim(1e-23, 1e3); ax.set_yticks([1e-20, 1e-16, 1e-12, 1e-8, 1e-4, 1])
ax.yaxis.set_major_formatter(sci); ax.minorticks_off()
ax.set_xticks([4, 10, 20, 30, 40, 50]); ax.set_xlim(2, 58)
ax.set_xlabel("깊이 (은닉층의 수. 잔차 연결에서는 Linear 층의 수)", fontsize=9)
ax.set_ylabel("첫 층 가중치 기울기의 크기 (로그 축)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="lower left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_4_9.png", dpi=300)
plt.show()

In [ ]:
# 그림 4-10
rows = [(20, "기본값"), (20, "1/sqrt(m)"), (20, "sqrt(2/m)"), (20, "기본값+정규화"), (20, "잔차(기본값 시작)"), (20, "잔차(0 시작)"),
        (50, "1/sqrt(m)"), (50, "sqrt(2/m)"), (50, "기본값+정규화"), (50, "잔차(기본값 시작)"), (50, "잔차(0 시작)")]
shallow = sorted(mark for loss, mark in grid[1, "기본값"])
fig, ax = plt.subplots(figsize=(7, 5.4))
for pos, (depth, name) in enumerate(rows):
    y = len(rows) - pos + (1 if depth == 20 else 0)
    marks = [mark for loss, mark in final[depth, name]]
    good = [mark for mark in marks if np.isfinite(mark)]
    ax.scatter(good, [y] * len(good), s=34, color=TEAL if max(good, default=9) < 3.5 else BRICK, edgecolor="white", lw=0.6, zorder=4)
    if len(good) < 5:
        ax.text(7.05, y, f"발산 {5 - len(good)}번", fontsize=9, color=BRICK, va="center")
    ax.text(2.72, y, f"{depth}층  {name}", fontsize=9, color="#3a4350", ha="right", va="center")
    if len(good) == 5:
        ax.text(7.75, y, f"{np.median(good):.2f}", fontsize=9, color=TEAL if max(good) < 3.5 else BRICK, va="center", weight="bold")
ax.axvline(shallow[1], color=NAVY, lw=1.0, ls=(0, (5, 2)))
ax.text(shallow[1] + 0.04, 14.0, f"은닉층 1개, 유닛 32개: {shallow[1]:.2f}GW", fontsize=9, color=NAVY, va="center")
ax.axvline(6.58, color=GRAY, lw=1.0, ls=(0, (2, 2)))
ax.text(6.54, 13.2, "평균으로 예측: 6.58GW", fontsize=9, color="#5b6470", ha="right", va="center")
ax.axhline(6.5, color="#c9d1da", lw=0.8)
ax.text(7.75, 13.2, "중앙값", fontsize=9, color="#3a4350", va="center")
ax.set_xlim(2.8, 8.3); ax.set_ylim(0.3, 14.6); ax.set_yticks([])
ax.set_xticks([3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5])
ax.set_xlabel("2025년 RMSE (GW). 점 하나가 시드 하나다", fontsize=9)
frame(ax, grid="x")
ax.spines["left"].set_visible(False)
fig.subplots_adjust(left=0.3, right=0.97, top=0.97, bottom=0.1)
fig.savefig("fig_4_10.png", dpi=300)
plt.show()